# Table Parsing Tool — Demo & Testing

This notebook demonstrates the **parse_table** tool for agentic extraction.

The tool provides deterministic Markdown table parsing that the extraction agent
can use to efficiently extract structured tabular data from OCR output without
LLM inference. The agent decides when to use this tool based on table quality
and OCR confidence metrics.

**No AWS credentials or S3 required** — this notebook tests the parser directly.

## Contents
1. Basic table parsing
2. Multi-table detection
3. OCR confidence integration
4. Quality metrics & expected columns
5. Realistic bank statement example
6. Nuveen PDF simulation (financial tables)
7. End-to-end with agentic extraction (requires Bedrock)

In [1]:
import sys
from pathlib import Path

ROOTDIR = Path("../..").resolve()
# Add idp_common to path if not installed
idp_common_path = str(ROOTDIR / "lib" / "idp_common_pkg")
if idp_common_path not in sys.path:
    sys.path.insert(0, idp_common_path)

# Uncomment the line below if idp_common is not already installed in your kernel:
# %pip install -q -e "{idp_common_path}[agents]"
print(f"ROOTDIR: {ROOTDIR}")
print(f"idp_common path: {idp_common_path}")

ROOTDIR: /Users/strahanr/Projects/idp2
idp_common path: /Users/strahanr/Projects/idp2/lib/idp_common_pkg


In [2]:
import json
from idp_common.extraction.tools.table_parser import (
    parse_markdown_tables,
    create_parse_table_tool,
)
print("✅ Table parser imported successfully")

✅ Table parser imported successfully


## 1. Basic Table Parsing

In [3]:
simple_table = """
| Name | Age | City |
|---|---|---|
| Alice | 30 | New York |
| Bob | 25 | Los Angeles |
| Charlie | 35 | Chicago |
"""

result = parse_markdown_tables(simple_table)
print(f"Status: {result['status']}")
print(f"Columns: {result['columns']}")
print(f"Row count: {result['row_count']}")
print(f"\nParsed rows:")
for row in result['rows']:
    print(f"  {row}")
print(f"\nQuality: {json.dumps(result['quality'], indent=2)}")

Status: success
Columns: ['Name', 'Age', 'City']
Row count: 3

Parsed rows:
  {'Name': 'Alice', 'Age': '30', 'City': 'New York'}
  {'Name': 'Bob', 'Age': '25', 'City': 'Los Angeles'}
  {'Name': 'Charlie', 'Age': '35', 'City': 'Chicago'}

Quality: {
  "parse_success_rate": 1.0,
  "consistent_column_count": true,
  "empty_cell_count": 0,
  "total_cells": 9,
  "empty_cell_ratio": 0.0,
  "avg_confidence": null,
  "low_confidence_cells": [],
  "confidence_available": false
}


## 2. Multi-Table Detection

In [4]:
multi_table_text = """
Document Title: Financial Report Q4 2024

Revenue Summary:
| Quarter | Revenue | Growth |
|---|---|---|
| Q1 | $1.2M | 5% |
| Q2 | $1.5M | 8% |
| Q3 | $1.8M | 12% |
| Q4 | $2.1M | 15% |

Expense Breakdown:
| Category | Amount | % of Total |
|---|---|---|
| Payroll | $800K | 45% |
| Operations | $400K | 22% |
| Marketing | $300K | 17% |
| R&D | $280K | 16% |
"""

result = parse_markdown_tables(multi_table_text)
print(f"Tables found: {result['table_count']}")
for i, table in enumerate(result['tables']):
    print(f"\n--- Table {i} ---")
    print(f"  Columns: {table['columns']}")
    print(f"  Rows: {table['row_count']}")
    for row in table['rows']:
        print(f"    {row}")

Tables found: 2

--- Table 0 ---
  Columns: ['Quarter', 'Revenue', 'Growth']
  Rows: 4
    {'Quarter': 'Q1', 'Revenue': '$1.2M', 'Growth': '5%'}
    {'Quarter': 'Q2', 'Revenue': '$1.5M', 'Growth': '8%'}
    {'Quarter': 'Q3', 'Revenue': '$1.8M', 'Growth': '12%'}
    {'Quarter': 'Q4', 'Revenue': '$2.1M', 'Growth': '15%'}

--- Table 1 ---
  Columns: ['Category', 'Amount', '% of Total']
  Rows: 4
    {'Category': 'Payroll', 'Amount': '$800K', '% of Total': '45%'}
    {'Category': 'Operations', 'Amount': '$400K', '% of Total': '22%'}
    {'Category': 'Marketing', 'Amount': '$300K', '% of Total': '17%'}
    {'Category': 'R&D', 'Amount': '$280K', '% of Total': '16%'}


## 3. OCR Confidence Integration

In [5]:
# Simulate OCR text with a table
ocr_table = """
| Date | Description | Amount |
|---|---|---|
| 01/15/2024 | Direct Deposit | 3,500.00 |
| 01/16/2024 | ATM Withdrawal | -200.00 |
| O1/18/2024 | Online Transfer | -1,5OO.00 |
"""

# Simulate Textract confidence data (same format as textConfidence.json)
confidence_data = """| Text | Confidence |
|:---|---:|
| 01/15/2024 | 99.8 |
| Direct Deposit | 99.5 |
| 3,500.00 | 98.2 |
| 01/16/2024 | 99.1 |
| ATM Withdrawal | 97.8 |
| -200.00 | 99.0 |
| O1/18/2024 | 72.3 |
| Online Transfer | 96.5 |
| -1,5OO.00 | 68.9 |"""

result = parse_markdown_tables(ocr_table, confidence_data=confidence_data)

print(f"Status: {result['status']}")
print(f"Average confidence: {result['quality'].get('avg_confidence')}")
print(f"\n⚠️ Low confidence cells (likely OCR errors):")
for cell in result['quality'].get('low_confidence_cells', []):
    print(f"  Row {cell['row']}, Column '{cell['column']}': "
          f"'{cell['cell_value']}' (confidence: {cell['confidence']})")

print(f"\n💡 The agent would use LLM reasoning to verify/correct these low-confidence cells")
print(f"   'O1/18/2024' is likely '01/18/2024' (O→0 OCR error)")
print(f"   '-1,5OO.00' is likely '-1,500.00' (O→0 OCR error)")

Status: success
Average confidence: 92.34

⚠️ Low confidence cells (likely OCR errors):
  Row 2, Column 'Date': 'O1/18/2024' (confidence: 72.3)
  Row 2, Column 'Amount': '-1,5OO.00' (confidence: 68.9)

💡 The agent would use LLM reasoning to verify/correct these low-confidence cells
   'O1/18/2024' is likely '01/18/2024' (O→0 OCR error)
   '-1,5OO.00' is likely '-1,500.00' (O→0 OCR error)


## 4. Expected Columns Validation

In [6]:
table_text = """
| Dt | Desc | Amt | Bal |
|---|---|---|---|
| 01/15 | Deposit | 3500 | 5200 |
| 01/16 | ATM | -200 | 5000 |
"""

# Check if parsed columns match expected schema fields
result = parse_markdown_tables(
    table_text,
    expected_columns=["Date", "Description", "Amount", "Balance"]
)

column_match = result['quality']['column_match']
print(f"Actual columns: {result['columns']}")
print(f"Expected: ['Date', 'Description', 'Amount', 'Balance']")
print(f"\nMatch ratio: {column_match['match_ratio']:.0%}")
print(f"Missing expected: {column_match['missing_expected']}")
print(f"Extra actual: {column_match['extra_actual']}")
print(f"\n💡 The agent would map: Dt→Date, Desc→Description, Amt→Amount, Bal→Balance")

Actual columns: ['Dt', 'Desc', 'Amt', 'Bal']
Expected: ['Date', 'Description', 'Amount', 'Balance']

Match ratio: 0%
Missing expected: ['amount', 'balance', 'date', 'description']
Extra actual: ['amt', 'bal', 'desc', 'dt']

💡 The agent would map: Dt→Date, Desc→Description, Amt→Amount, Bal→Balance


## 5. Realistic Bank Statement

In [7]:
bank_statement = """Account Number: 12345678
Statement Period: January 1, 2024 - January 31, 2024
Account Holder: John Smith

| Date | Description | Debit | Credit | Balance |
|---|---|---|---|---|
| 01/01/2024 | Opening Balance | | | 5,000.00 |
| 01/03/2024 | Direct Deposit - ACME Corp | | 3,500.00 | 8,500.00 |
| 01/05/2024 | Check #1234 - Rent | 1,200.00 | | 7,300.00 |
| 01/08/2024 | POS Purchase - Grocery Store | 156.32 | | 7,143.68 |
| 01/10/2024 | Electric Bill Payment | 89.50 | | 7,054.18 |
| 01/12/2024 | ATM Withdrawal | 200.00 | | 6,854.18 |
| 01/15/2024 | Internet Banking Transfer | 500.00 | | 6,354.18 |
| 01/17/2024 | Direct Deposit - ACME Corp | | 3,500.00 | 9,854.18 |
| 01/20/2024 | Auto Insurance Premium | 175.00 | | 9,679.18 |
| 01/22/2024 | Gas Station | 45.67 | | 9,633.51 |
| 01/25/2024 | Online Purchase - Amazon | 89.99 | | 9,543.52 |
| 01/28/2024 | Water Bill | 42.15 | | 9,501.37 |
| 01/31/2024 | Closing Balance | | | 9,501.37 |

Closing Balance: $9,501.37
Total Deposits: $7,000.00
Total Withdrawals: $2,498.63"""

result = parse_markdown_tables(bank_statement)

print(f"✅ Parsed {result['row_count']} transactions")
print(f"   Columns: {result['columns']}")
print(f"   Quality: parse_success_rate={result['quality']['parse_success_rate']:.2f}")
print(f"   Empty cells: {result['quality']['empty_cell_count']} (normal for debit/credit columns)")

print(f"\n📊 Transaction Data:")
print(f"{'Date':<14} {'Description':<35} {'Debit':>10} {'Credit':>10} {'Balance':>12}")
print("-" * 85)
for row in result['rows']:
    print(f"{row['Date']:<14} {row['Description']:<35} {row['Debit']:>10} {row['Credit']:>10} {row['Balance']:>12}")

✅ Parsed 13 transactions
   Columns: ['Date', 'Description', 'Debit', 'Credit', 'Balance']
   Quality: parse_success_rate=1.00
   Empty cells: 15 (normal for debit/credit columns)

📊 Transaction Data:
Date           Description                              Debit     Credit      Balance
-------------------------------------------------------------------------------------
01/01/2024     Opening Balance                                               5,000.00
01/03/2024     Direct Deposit - ACME Corp                       3,500.00     8,500.00
01/05/2024     Check #1234 - Rent                    1,200.00                7,300.00
01/08/2024     POS Purchase - Grocery Store            156.32                7,143.68
01/10/2024     Electric Bill Payment                    89.50                7,054.18
01/12/2024     ATM Withdrawal                          200.00                6,854.18
01/15/2024     Internet Banking Transfer               500.00                6,354.18
01/17/2024     Direct Dep

## 6. Nuveen-Style Financial Table Simulation

The `samples/Nuveen.pdf` contains a multi-page financial document with 500+ rows of
fund distribution data in tables. Below we simulate the OCR output format.

In [8]:
# Simulate a portion of what Textract would produce from the Nuveen PDF
nuveen_ocr = """Nuveen Estimated Capital Gains Distributions
Estimated 2024 Capital Gains Distributions for Nuveen Mutual Funds

| Fund Name | Ticker | Record Date | Ex-Dividend Date | Payment Date | Est Short-Term Capital Gains | Est Long-Term Capital Gains | NAV as of 10/31/2024 | Total Cap Gain Dist % of NAV |
|---|---|---|---|---|---|---|---|---|
| Nuveen Large Cap Value Fund | NNGAX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.0000 | $1.2345 | $32.45 | 3.80% |
| Nuveen Large Cap Growth Opportunities Fund | NROGX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.1500 | $2.8900 | $45.67 | 6.65% |
| Nuveen Mid Cap Value Fund | FASEX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.0000 | $0.7800 | $28.90 | 2.70% |
| Nuveen Small Cap Value Fund | FSCAX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.2100 | $1.5600 | $35.12 | 5.04% |
| Nuveen International Fund | NIFAX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.0000 | $0.0000 | $18.23 | 0.00% |
| Nuveen Dividend Value Fund | FFEIX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.0300 | $0.9500 | $22.56 | 4.34% |
| Nuveen Growth Fund | NSAGX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.5000 | $3.2100 | $52.34 | 7.09% |
| Nuveen Real Estate Securities Fund | NREAX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.0000 | $0.4200 | $15.67 | 2.68% |
| Nuveen All Cap Growth Opportunities Fund | NAGOX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.1200 | $1.8900 | $41.23 | 4.88% |
| Nuveen Equity Market Neutral Fund | NMEAX | 12/13/2024 | 12/16/2024 | 12/17/2024 | $0.0000 | $0.0000 | $10.05 | 0.00% |

Important: These are estimated distributions and may change."""

result = parse_markdown_tables(nuveen_ocr)

print(f"✅ Parsed {result['row_count']} fund records")
print(f"   Columns ({len(result['columns'])}): {result['columns']}")
print(f"   Parse success rate: {result['quality']['parse_success_rate']:.2f}")
print(f"   Consistent columns: {result['quality']['consistent_column_count']}")

print(f"\n📊 Fund Distribution Data (first 5):")
for i, row in enumerate(result['rows'][:5]):
    print(f"  [{i}] {row['Fund Name'][:40]:<42} Ticker: {row['Ticker']:<8} "
          f"ST: {row['Est Short-Term Capital Gains']:>10} "
          f"LT: {row['Est Long-Term Capital Gains']:>10} "
          f"NAV: {row['NAV as of 10/31/2024']:>8} "
          f"Dist%: {row['Total Cap Gain Dist % of NAV']:>6}")

print(f"\n💡 This is what the agent does with parse_table output:")
print(f"   1. Maps 'Fund Name' → schema field 'fund_name'")
print(f"   2. Maps 'Est Short-Term Capital Gains' → 'estimated_short_term_capital_gains'")
print(f"   3. Converts '$32.45' → 32.45 for float fields")
print(f"   4. Stores via extraction_tool with Pydantic validation")

✅ Parsed 10 fund records
   Columns (9): ['Fund Name', 'Ticker', 'Record Date', 'Ex-Dividend Date', 'Payment Date', 'Est Short-Term Capital Gains', 'Est Long-Term Capital Gains', 'NAV as of 10/31/2024', 'Total Cap Gain Dist % of NAV']
   Parse success rate: 1.00
   Consistent columns: True

📊 Fund Distribution Data (first 5):
  [0] Nuveen Large Cap Value Fund                Ticker: NNGAX    ST:    $0.0000 LT:    $1.2345 NAV:   $32.45 Dist%:  3.80%
  [1] Nuveen Large Cap Growth Opportunities Fu   Ticker: NROGX    ST:    $0.1500 LT:    $2.8900 NAV:   $45.67 Dist%:  6.65%
  [2] Nuveen Mid Cap Value Fund                  Ticker: FASEX    ST:    $0.0000 LT:    $0.7800 NAV:   $28.90 Dist%:  2.70%
  [3] Nuveen Small Cap Value Fund                Ticker: FSCAX    ST:    $0.2100 LT:    $1.5600 NAV:   $35.12 Dist%:  5.04%
  [4] Nuveen International Fund                  Ticker: NIFAX    ST:    $0.0000 LT:    $0.0000 NAV:   $18.23 Dist%:  0.00%

💡 This is what the agent does with parse_table outp

## 7. Using the Tool as a Strands @tool (Agent Integration)

In [9]:
# Create the tool as it would be created inside the agentic extraction agent
confidence_data_by_page = {
    "1": """| Text | Confidence |
|:---|---:|
| Nuveen Large Cap Value Fund | 99.5 |
| NNGAX | 99.8 |
| 12/13/2024 | 99.2 |
| $0.0000 | 98.5 |
| $1.2345 | 97.8 |
| $32.45 | 99.1 |
| 3.80% | 98.9 |"""
}

parse_table_tool = create_parse_table_tool(
    confidence_data_by_page=confidence_data_by_page
)

print(f"Tool created: {parse_table_tool.__name__}")
print(f"Tool docstring: {parse_table_tool.__doc__[:200]}...")
print(f"\n✅ This tool is registered in the agent's tool list when")
print(f"   extraction.agentic.table_parsing.enabled = true")

Tool created: parse_table
Tool docstring: 
        Parse a Markdown table from OCR text into structured rows and columns.

        Use this tool when you identify a well-formatted table in the document text.
        It parses the table determ...

✅ This tool is registered in the agent's tool list when
   extraction.agentic.table_parsing.enabled = true


## 8. Configuration

The table parsing tool is configured via the standard IDP config system:

In [10]:
from idp_common.config.models import IDPConfig

# Default: disabled
config = IDPConfig()
tp = config.extraction.agentic.table_parsing
print(f"Default config:")
print(f"  enabled: {tp.enabled}")
print(f"  min_confidence_threshold: {tp.min_confidence_threshold}")
print(f"  min_parse_success_rate: {tp.min_parse_success_rate}")
print(f"  use_confidence_data: {tp.use_confidence_data}")

# Enable with custom thresholds
config = IDPConfig(extraction={
    "agentic": {
        "enabled": True,
        "table_parsing": {
            "enabled": True,
            "min_confidence_threshold": 90.0,
            "min_parse_success_rate": 0.85,
        }
    }
})
tp = config.extraction.agentic.table_parsing
print(f"\nCustom config:")
print(f"  enabled: {tp.enabled}")
print(f"  min_confidence_threshold: {tp.min_confidence_threshold}")
print(f"  min_parse_success_rate: {tp.min_parse_success_rate}")

print(f"\n💡 These settings are also visible and editable in the Web UI")
print(f"   under Extraction → Agentic Extraction → Table Parsing Tool")

Default config:
  enabled: False
  min_confidence_threshold: 95.0
  min_parse_success_rate: 0.9
  use_confidence_data: True

Custom config:
  enabled: True
  min_confidence_threshold: 90.0
  min_parse_success_rate: 0.85

💡 These settings are also visible and editable in the Web UI
   under Extraction → Agentic Extraction → Table Parsing Tool


## 9. End-to-End with Nuveen.pdf (Requires Bedrock Access)

This cell uses the agentic extraction directly with the Nuveen PDF.
**Requires**: AWS credentials with Bedrock access to Anthropic Claude.

In [11]:
# Uncomment to run — requires Bedrock access
"""
import asyncio
from pathlib import Path
from pydantic import BaseModel, Field
from strands.types.content import ContentBlock, Message
from strands.types.media import DocumentContent
from idp_common.extraction.agentic_idp import structured_output_async
from idp_common.config.models import IDPConfig


# Define schema matching Nuveen table structure
class FundRow(BaseModel):
    fund_name: str
    ticker: str
    record_date: str
    ex_dividend_date: str
    payment_date: str
    estimated_short_term_capital_gains: str
    estimated_long_term_capital_gains: str
    nav_as_of_10_31_2024: float
    total_cap_gain_distribution_prc_of_nav: float

class NuveenDocument(BaseModel):
    document_name: str
    table_rows: list[FundRow] = Field(min_length=10)


# Load PDF
pdf_path = Path(ROOTDIR) / "samples" / "Nuveen.pdf"
with open(pdf_path, "rb") as f:
    pdf_data = f.read()

# Config with table parsing ENABLED
config = IDPConfig(extraction={
    "agentic": {
        "enabled": True,
        "table_parsing": {
            "enabled": True,
            "min_confidence_threshold": 95.0,
        }
    }
})

result, response = await structured_output_async(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    data_format=NuveenDocument,
    prompt=Message(
        role="user",
        content=[
            ContentBlock(text="Extract all fund distribution data from this document"),
            ContentBlock(
                document=DocumentContent(
                    format="pdf",
                    source={"bytes": pdf_data},
                    name="Nuveen.pdf",
                )
            ),
        ],
    ),
    config=config,
)

print(f"✅ Extracted {len(result.table_rows)} fund rows")
print(f"   Document: {result.document_name}")
for row in result.table_rows[:5]:
    print(f"   {row.fund_name[:40]:<42} {row.ticker:<8} NAV: ${row.nav_as_of_10_31_2024:.2f}")

# Check metering (token usage)
metering = response.get('metering', {})
for key, usage in metering.items():
    print(f"\nToken usage ({key}):")
    print(f"  Input: {usage.get('inputTokens', 0):,}")
    print(f"  Output: {usage.get('outputTokens', 0):,}")
    print(f"  Cache read: {usage.get('cacheReadInputTokens', 0):,}")
"""
print("☝️ Uncomment the cell above and run to test with actual Bedrock + Nuveen.pdf")

☝️ Uncomment the cell above and run to test with actual Bedrock + Nuveen.pdf


## Summary

The `parse_table` tool provides:
- **Deterministic parsing** — no LLM tokens used for well-structured tables
- **Quality metrics** — parse_success_rate, empty_cell_ratio, confidence scores
- **OCR confidence integration** — flags low-confidence cells for LLM verification
- **Multi-table detection** — finds and parses all tables in OCR text
- **Column validation** — compares against expected schema field names

The agent's workflow:
1. Sees Markdown tables in document text
2. Calls `parse_table` for fast deterministic parsing
3. Reviews quality metrics
4. If good: maps columns → schema fields, stores via `extraction_tool`
5. If poor: falls back to LLM extraction for those sections
6. For low-confidence cells: uses LLM reasoning to verify/correct specific values